In [9]:
import math
import numpy as np
import random
import os
import csv
from concurrent.futures import ProcessPoolExecutor, as_completed
from datetime import datetime


class AntQ:

    def __init__(self, distance_matrix, m=None,
                 alpha=0.1, gamma=0.3,
                 delta=1, beta=2,
                 q0=0.9, W=10):

        self.distances = np.array(distance_matrix, dtype=float)
        np.fill_diagonal(self.distances, np.inf)

        self.n = len(self.distances)
        self.m = m if m is not None else self.n

        self.alpha = alpha
        self.gamma = gamma
        self.delta = delta
        self.beta = beta
        self.q0 = q0
        self.W = W

        with np.errstate(divide='ignore'):
            self.HE = 1.0 / self.distances

        avg_length = np.mean(
            self.distances[self.distances != np.inf]
        )

        self.AQ0 = 1.0 / (avg_length * self.n)
        self.AQ = np.full((self.n, self.n), self.AQ0)

    def _action_choice(self, r, unvisited):

        q = random.random()

        aq_vals = self.AQ[r, unvisited]
        he_vals = self.HE[r, unvisited]

        evaluations = (
            aq_vals ** self.delta
        ) * (
            he_vals ** self.beta
        )

        if q <= self.q0:
            return unvisited[np.argmax(evaluations)]

        total = np.sum(evaluations)

        if total <= 0 or np.isnan(total):
            return random.choice(unvisited)

        probabilities = evaluations / total
        return np.random.choice(unvisited, p=probabilities)

    def run(self, iterations, patience=None):

        best_overall_tour = None
        best_overall_length = float("inf")
        no_improve = 0

        for _ in range(iterations):

            tours = []
            unvisited_sets = []
            current_cities = []

            for _ in range(self.m):

                start = random.randint(0, self.n - 1)
                tours.append([start])
                current_cities.append(start)

                unvisited = list(range(self.n))
                unvisited.remove(start)
                unvisited_sets.append(unvisited)

            for step in range(self.n):

                for k in range(self.m):

                    r = current_cities[k]

                    if step < self.n - 1:
                        s = self._action_choice(
                            r,
                            unvisited_sets[k]
                        )
                        unvisited_sets[k].remove(s)
                    else:
                        s = tours[k][0]

                    tours[k].append(s)
                    current_cities[k] = s

                    if step < self.n - 1 and unvisited_sets[k]:
                        max_aq = max(
                            self.AQ[s, z]
                            for z in unvisited_sets[k]
                        )
                    else:
                        max_aq = 0

                    self.AQ[r, s] = (
                        (1 - self.alpha) * self.AQ[r, s]
                        + self.alpha * self.gamma * max_aq
                    )

            lengths = [
                sum(
                    self.distances[t[i], t[i + 1]]
                    for i in range(self.n)
                )
                for t in tours
            ]

            best_idx = np.argmin(lengths)
            iter_best_tour = tours[best_idx]
            iter_best_length = lengths[best_idx]

            if iter_best_length < best_overall_length:
                best_overall_length = iter_best_length
                best_overall_tour = iter_best_tour
                no_improve = 0
            else:
                no_improve += 1

            delta_AQ = self.W / iter_best_length

            for i in range(self.n):
                r = iter_best_tour[i]
                s = iter_best_tour[i + 1]

                self.AQ[r, s] = (
                    (1 - self.alpha) * self.AQ[r, s]
                    + self.alpha * delta_AQ
                )

            if patience and no_improve >= patience:
                break

        return best_overall_tour, best_overall_length


def geo_to_radians(x):
    deg = int(x)
    minutes = x - deg
    return math.pi * (deg + 5.0 * minutes / 3.0) / 180.0


def geo_distance(a, b):

    RRR = 6378.388

    lat1 = geo_to_radians(a[0])
    lon1 = geo_to_radians(a[1])
    lat2 = geo_to_radians(b[0])
    lon2 = geo_to_radians(b[1])

    q1 = math.cos(lon1 - lon2)
    q2 = math.cos(lat1 - lat2)
    q3 = math.cos(lat1 + lat2)

    dij = RRR * math.acos(
        0.5 * ((1 + q1) * q2 - (1 - q1) * q3)
    ) + 1.0

    return int(dij)


def build_geo_matrix(coords):

    n = len(coords)
    M = np.zeros((n, n))

    for i in range(n):
        for j in range(n):
            if i != j:
                M[i][j] = geo_distance(
                    coords[i],
                    coords[j]
                )
            else:
                M[i][j] = np.inf

    return M


def build_euc_matrix(coords):

    n = len(coords)
    M = np.zeros((n, n))

    for i in range(n):
        for j in range(n):
            if i != j:
                M[i][j] = round(math.sqrt(
                    (coords[i][0] - coords[j][0])**2 +
                    (coords[i][1] - coords[j][1])**2
                ))
            else:
                M[i][j] = np.inf

    return M


def load_tsplib(filepath):

    coords = []
    edge_type = "EUC_2D"
    reading = False

    with open(filepath) as f:
        for line in f:

            line = line.strip()

            if line.startswith("EDGE_WEIGHT_TYPE"):
                edge_type = line.split(":")[1].strip()

            if line == "NODE_COORD_SECTION":
                reading = True
                continue

            if line == "EOF":
                break

            if reading:
                p = line.split()
                coords.append(
                    (float(p[1]), float(p[2]))
                )

    if edge_type == "GEO":
        return build_geo_matrix(coords)

    return build_euc_matrix(coords)


def run_trial(trial, instance_name, distance_matrix):

    start = datetime.now()

    n = len(distance_matrix)
    antq = AntQ(distance_matrix, m=n)

    iterations = 10 * n
    patience = 2 * n

    _, best_length = antq.run(
        iterations,
        patience
    )

    end = datetime.now()

    return (
        instance_name,
        "Ant-Q",
        trial,
        best_length,
        (end - start).total_seconds()
    )


instances_dir = "NewInstances"

instance_files = [
    os.path.join(instances_dir, f)
    for f in os.listdir(instances_dir)
    if f.endswith(".tsp")
]

os.makedirs("Results", exist_ok=True)

csv_file = "Results/AntQ_Results.csv"

with open(csv_file, "w", newline="") as file:

    writer = csv.writer(file)

    writer.writerow([
        "Instance",
        "Algorithm",
        "Trial",
        "PathLength",
        "TimeSeconds"
    ])

    for instance_file in instance_files:

        instance_name = os.path.basename(
            instance_file
        ).replace(".tsp", "")

        print("\n==============================")
        print("Processing:", instance_name)

        distance_matrix = load_tsplib(
            instance_file
        )

        num_trials = 20
        results = []

        with ProcessPoolExecutor(
                max_workers=num_trials) as executor:

            futures = [
                executor.submit(
                    run_trial,
                    trial,
                    instance_name,
                    distance_matrix
                )
                for trial in range(1, num_trials + 1)
            ]

            for future in as_completed(futures):
                results.append(
                    future.result()
                )

        results.sort(key=lambda x: x[2])

        for row in results:
            writer.writerow(row)

print("\nSaved results at:", csv_file)


Processing: gr96

Processing: rat99

Saved results at: Results/AntQ_Results.csv
